# 1 · Quickstart

This notebook runs the NMC111 half-cell models from Python. Each model is a porous NMC111 cathode against a lithium-foil counter electrode, with a separator between them. There are two particle models:

- **uniform**: each electrode node holds uniform-concentration particles;
- **agglomerate**: each electrode node holds porous agglomerates of small crystals, with their own electrolyte, conduction and kinetics inside (docs/model.md section 6).

The same models are available as self-contained Fortran (`fortran/nmc.f90`) and C++ (`cpp/nmc.cpp`) programs, which read the same input files.

All parameters are generic defaults for illustration (docs/parameters.md). They are not fitted to any particular cell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nmc_model.inputs import load
from nmc_model.agglomerate import AggParams
from nmc_model.agglomerate_corrected import run_corrected
from nmc_model.uniform.params import Params
from nmc_model.uniform.simulate import run as run_uniform

## Input files

`input/uniform.nml` and `input/agglomerate.nml` list every parameter at its default. `load` reads them into parameter objects; `python -m nmc_model input/agglomerate.nml` runs one and writes `Time_Voltage.txt`.

In [ ]:
p_u, model, _ = load("../input/uniform.nml")
p_a, model_a, _ = load("../input/agglomerate.nml")
print(model, "C-rate", p_u.C_rate, "| thickness", p_u.L_cath_um, "um")
print(model_a, "C-rate", p_a.C_rate, "| thickness", p_a.thickness_um, "um, loading", p_a.mass_loading * 1e3, "mg/cm2")

## A 1C discharge with each model

The agglomerate model is run here on a coarser grid than the input file's (30 electrode nodes, 12 along each agglomerate radius, 5 s steps) so that it takes a few seconds. Both runs stop at their lower voltage cutoff.

In [ ]:
r_u = run_uniform(Params(mode="corrected", C_rate=1.0))
coarse = dict(nj=30, nja=12, dt_s=5.0)
r_a = run_corrected(AggParams(mode="corrected", C_rate=1.0, **coarse))
print("uniform:", r_u.exit_reason, "| agglomerate:", r_a.exit_reason)

def mAhg(r, M):
    return r.array[:, 2] * 96485.0 / (M * 3.6)          # electron equivalents -> mAh/g

plt.plot(mAhg(r_u, p_u.M), r_u.array[:, 1], label="uniform particles")
plt.plot(mAhg(r_a, p_a.M), r_a.array[:, 1], label="agglomerates")
plt.xlabel("capacity [mAh/g]"); plt.ylabel("cell voltage vs Li [V]"); plt.legend(); plt.title("1C discharge")
plt.show()

The two models have different open-circuit-potential fits and electrode designs (from the original studies), so the curves are not a like-for-like comparison. Notebook 2 looks inside the agglomerate model.

## The output file

Columns written in corrected mode (docs/protocol.md): state, time [h], cell voltage against the lithium foil [V], electron equivalents, the foil overpotential [mV], its exchange current [mA/cm²], the electrolyte concentration at the foil, the current [mA/cm²], the protocol step, and the foil's Nernst potential [mV]. The agglomerate model adds the surface lithiation of the agglomerate next to the separator (`x_front`) and the electrolyte concentration at the current collector.

In [ ]:
import os, tempfile
path = os.path.join(tempfile.mkdtemp(), "Time_Voltage.txt")
r_a.write(path)
print(open(path).read().splitlines()[0])
print(open(path).read().splitlines()[2])

## Protocols

A protocol is a list of steps: constant current (`cc`), constant voltage (`cv`) and rest. Here: discharge at 1C, rest 10 minutes, charge at 1C to 4.3 V, hold 4.3 V until the current falls to C/20, rest.

In [ ]:
cycle = "cc C=1 Vmin=3.0; rest t=600; cc C=-1 Vmax=4.3; cv V=4.3 Imin=0.05; rest t=600"
r = run_corrected(AggParams(mode="corrected", steps=cycle, **coarse))
a = r.array
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(a[:, 0], a[:, 1]); ax[0].set_ylabel("V [V]")
ax[1].plot(a[:, 0], a[:, 6]); ax[1].set_ylabel("I [mA/cm²]"); ax[1].set_xlabel("time [h]")
plt.show()
print(r.exit_reason)